In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer, make_column_transformer, make_column_selector
from sklearn.metrics import r2_score, log_loss, accuracy_score, roc_auc_score, log_loss

from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, ElasticNet
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.ensemble import VotingClassifier, VotingRegressor

from sklearn.ensemble import BaggingClassifier, BaggingRegressor

from tqdm import tqdm 
import warnings
warnings.filterwarnings("ignore")

In [4]:
concrete = pd.read_csv('D:/Machine_Learning/Cases/Concrete_Strength/Concrete_Data.csv')

X = concrete.drop('Strength',axis=1)
y=concrete['Strength']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26)

In [5]:
dtr = DecisionTreeRegressor(random_state=26)
knn = KNeighborsRegressor()
lr = LinearRegression()
el = ElasticNet()

estims = [dtr, knn, lr, el]
scores = []

for e in tqdm(estims):
    for n in [10,15,50,100]:
        bagg = BaggingRegressor( random_state=26, estimator=e, n_estimators=n)
        bagg.fit(X_train, y_train)
        y_pred = bagg.predict(X_test)
        scores.append([e, n, r2_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores, columns=['E','n','r2_score'])
df_scores.sort_values('r2_score', ascending=False)

100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.59it/s]


,E,n,r2_score
3,DecisionTreeRegressor(random_state=26),100,0.898804
2,DecisionTreeRegressor(random_state=26),50,0.897347
0,DecisionTreeRegressor(random_state=26),10,0.894547
1,DecisionTreeRegressor(random_state=26),15,0.894110
7,KNeighborsRegressor(),100,0.683729
6,KNeighborsRegressor(),50,0.680985
4,KNeighborsRegressor(),10,0.678554
5,KNeighborsRegressor(),15,0.675296
12,ElasticNet(),10,0.620164
8,LinearRegression(),10,0.619629


# Road Accident

In [6]:
import os
os.chdir("C:/Users/PGCP-AI/Downloads/PredictingRoadAccident")

In [7]:
train = pd.read_csv('train.csv',index_col=0)
test = pd.read_csv('test.csv',index_col=0)

In [8]:
X = train.drop('accident_risk',axis=1)
y=train['accident_risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26)

In [9]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

trans = ColumnTransformer(transformers=[("OHE", ohe, make_column_selector(dtype_include=object))], remainder="passthrough",
                            verbose_feature_names_out=False).set_output(transform="pandas")

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

In [11]:
dtr = DecisionTreeRegressor(random_state=26)
bagg = BaggingRegressor( random_state=26, estimator=dtr, n_estimators=100)
bagg.fit(X_trn_ohe, y_train)
y_pred = bagg.predict(X_tst_ohe)
y_pred[y_pred<0]= 0
submit = pd.read_csv("sample_submission.csv")
submit['accident_risk'] = y_pred

submit.to_csv("sbt_bag_dtr_100.csv", index=False)

ValueError: Length of values (155327) does not match length of index (172585)